<a href="https://colab.research.google.com/github/nickkhymenko/asteria-credit-risk-ml/blob/main/03_feature_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03: Data Preparation and Feature Pipeline

**Project:** Asteria Finance, Repayment Difficulty Prediction (IB99Z0)

**CRISP-DM phase:** Data Preparation

This notebook turns the seven relational CSV files into **one row per applicant** (`SK_ID_CURR`) ready for modelling.

**Design decisions**

1. **SQL aggregation in DuckDB.** The history tables are large (over 1.3 million monthly bureau records and 1 million instalment records). DuckDB runs SQL directly on the CSV files, in-process and in parallel, so the joins and `GROUP BY`s are declarative, reproducible and scale far beyond what row-by-row pandas would comfortably handle. The same SQL would run on Spark or a data warehouse in production.
2. **Data quality fixes** found during EDA are applied here, once, so every model sees the same cleaned data:
    * `365243` is a placeholder for "no date" in the `DAYS_*` columns and is replaced with missing plus an indicator flag.
    * `CODE_GENDER = 'XNA'` is treated as missing.
    * Negative outstanding debt in the bureau table is floored at zero.
3. **"No history" is kept distinct from "clean history".** Applicants without records in a table get missing rates plus a `HAS_*` flag, because EDA showed that applicants with no bureau history are riskier ($10.3\%$) than those with a clean history ($7.6\%$).
4. **No target information is used in this notebook.** Every feature is computed from applicant data and historical records only, so there is no target leakage. Imputation and encoding that learn from the data happen later, *inside* the cross-validation folds in notebook 04.

In [ ]:
# Uncomment in a fresh environment (Colab already ships with most of these).
# %pip install -q duckdb pyarrow
import os
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
DATA_DIR = Path(os.environ.get("ASTERIA_DATA_DIR", "/content" if IN_COLAB else "data"))
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "applications": "loan_applications.csv",
    "previous": "previous_loan_applications.csv",
    "bureau": "external_credit_accounts.csv",
    "bureau_history": "external_account_history.csv",
    "instalments": "instalment_payments.csv",
    "pos": "pos_cash_balance.csv",
    "card": "credit_card_balance.csv",
}

# Unlabelled applicants we must score for predictions.csv.
# TODO: confirm the exact file name on my.wbs and update it here (or set ASTERIA_TEST_FILE).
TEST_FILE = os.environ.get("ASTERIA_TEST_FILE", "loan_applications_test.csv")

missing_files = [f for f in FILES.values() if not (DATA_DIR / f).exists()]
assert not missing_files, f"Missing files in {DATA_DIR.resolve()}: {missing_files}"
HAS_TEST = (DATA_DIR / TEST_FILE).exists()

DATE_SENTINEL = 365243  # placeholder used in the source system for "no date"

print("Data directory:", DATA_DIR.resolve())
print("Test file found:" if HAS_TEST else "Test file NOT found (predictions will be skipped):", TEST_FILE)

## 1. Register the history tables in DuckDB

Views are lazy: DuckDB reads each CSV only when a query needs it, and only the columns that query uses.

In [ ]:
con = duckdb.connect()

for name, fname in FILES.items():
    if name == "applications":
        continue
    path = (DATA_DIR / fname).as_posix()
    con.execute(
        f"CREATE OR REPLACE VIEW {name} AS "
        f"SELECT * FROM read_csv_auto('{path}', header = true, sample_size = -1)"
    )

for name in FILES:
    if name == "applications":
        continue
    n_rows, n_ids = con.execute(
        f"SELECT COUNT(*), COUNT(DISTINCT SK_ID_CURR) FROM {name}"
        if name != "bureau_history"
        else "SELECT COUNT(*), COUNT(DISTINCT SK_ID_BUREAU) FROM bureau_history"
    ).fetchone()
    print(f"{name:<15} rows = {n_rows:>10,}   distinct keys = {n_ids:>8,}")

## 2. Data quality checks before aggregating

These are the issues found during EDA (notebooks 01 and 02). We quantify them so the report can cite exact numbers.

In [ ]:
checks = {
    "previous: DAYS_* = 365243 placeholder (rows)": f'''
        SELECT COUNT(*) FROM previous
        WHERE DAYS_FIRST_DRAWING = {DATE_SENTINEL} OR DAYS_FIRST_DUE = {DATE_SENTINEL}
           OR DAYS_LAST_DUE_1ST_VERSION = {DATE_SENTINEL} OR DAYS_LAST_DUE = {DATE_SENTINEL}
           OR DAYS_TERMINATION = {DATE_SENTINEL}''',
    "bureau: negative outstanding debt (rows)": "SELECT COUNT(*) FROM bureau WHERE AMT_CREDIT_SUM_DEBT < 0",
    "bureau: accounts with no monthly history": '''
        SELECT COUNT(*) FROM bureau b
        WHERE NOT EXISTS (SELECT 1 FROM bureau_history h WHERE h.SK_ID_BUREAU = b.SK_ID_BUREAU)''',
    "instalments: missing payment date (rows)": "SELECT COUNT(*) FROM instalments WHERE DAYS_ENTRY_PAYMENT IS NULL",
    "card: negative balance (rows)": "SELECT COUNT(*) FROM card WHERE AMT_BALANCE < 0",
}
for label, sql in checks.items():
    print(f"{label:<50} {con.execute(sql).fetchone()[0]:>10,}")

## 3. Applicant-level aggregations (SQL)

Each query collapses a one-to-many table to one row per applicant. Features are chosen to capture **behaviour** (lateness, refusals, utilisation, delinquency), **exposure** (debt, limits) and **recency** (last 12 months), following the EDA findings.

In [ ]:
SQL_PREVIOUS = f'''
SELECT
    SK_ID_CURR,
    COUNT(*)                                                         AS PREV_COUNT,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS PREV_APPROVED_COUNT,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END) AS PREV_REFUSED_COUNT,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Canceled' THEN 1 ELSE 0 END) AS PREV_CANCELED_COUNT,
    AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1.0 ELSE 0 END) AS PREV_REFUSAL_RATE,
    SUM(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' AND DAYS_DECISION >= -365
             THEN 1 ELSE 0 END)                                      AS PREV_REFUSED_LAST_12M,
    AVG(AMT_CREDIT / NULLIF(AMT_APPLICATION, 0))                     AS PREV_CREDIT_TO_APPLIED_MEAN,
    MAX(AMT_CREDIT)                                                  AS PREV_AMT_CREDIT_MAX,
    AVG(AMT_ANNUITY)                                                 AS PREV_AMT_ANNUITY_MEAN,
    AVG(CNT_PAYMENT)                                                 AS PREV_TERM_MEAN,
    MAX(DAYS_DECISION)                                               AS PREV_DAYS_LAST_DECISION,
    AVG(CASE WHEN NAME_YIELD_GROUP = 'high' THEN 1.0 ELSE 0 END)     AS PREV_HIGH_YIELD_SHARE,
    AVG(NULLIF(DAYS_LAST_DUE, {DATE_SENTINEL})
        - NULLIF(DAYS_LAST_DUE_1ST_VERSION, {DATE_SENTINEL}))        AS PREV_LAST_DUE_SHIFT_MEAN
FROM previous
GROUP BY SK_ID_CURR
'''

SQL_BUREAU = '''
WITH agg AS (
    SELECT
        SK_ID_CURR,
        COUNT(*)                                                           AS BUR_COUNT,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)          AS BUR_ACTIVE_COUNT,
        SUM(CASE WHEN CREDIT_ACTIVE IN ('Sold', 'Bad debt') THEN 1 ELSE 0 END) AS BUR_SOLD_OR_BAD_COUNT,
        SUM(CASE WHEN CREDIT_DAY_OVERDUE > 0 THEN 1 ELSE 0 END)            AS BUR_OVERDUE_COUNT,
        MAX(CREDIT_DAY_OVERDUE)                                            AS BUR_MAX_DAYS_OVERDUE,
        MAX(AMT_CREDIT_MAX_OVERDUE)                                        AS BUR_MAX_AMT_OVERDUE,
        SUM(COALESCE(AMT_CREDIT_SUM_OVERDUE, 0))                           AS BUR_SUM_AMT_OVERDUE,
        SUM(CNT_CREDIT_PROLONG)                                            AS BUR_PROLONG_COUNT,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Active'
                 THEN GREATEST(COALESCE(AMT_CREDIT_SUM_DEBT, 0), 0) ELSE 0 END) AS BUR_ACTIVE_DEBT,
        SUM(CASE WHEN CREDIT_ACTIVE = 'Active'
                 THEN COALESCE(AMT_CREDIT_SUM, 0) ELSE 0 END)              AS BUR_ACTIVE_CREDIT,
        MAX(DAYS_CREDIT)                                                   AS BUR_DAYS_LAST_CREDIT,
        MIN(DAYS_CREDIT)                                                   AS BUR_DAYS_FIRST_CREDIT,
        SUM(CASE WHEN DAYS_CREDIT >= -365 THEN 1 ELSE 0 END)               AS BUR_NEW_LAST_12M,
        COUNT(DISTINCT CREDIT_TYPE)                                        AS BUR_CREDIT_TYPES
    FROM bureau
    GROUP BY SK_ID_CURR
)
SELECT *, BUR_ACTIVE_DEBT / NULLIF(BUR_ACTIVE_CREDIT, 0) AS BUR_ACTIVE_DEBT_RATIO
FROM agg
'''

# Monthly bureau history: first collapse to one row per account, then to one row per applicant.
SQL_BUREAU_HISTORY = '''
WITH h AS (
    SELECT SK_ID_BUREAU, MONTHS_BALANCE, CAST(STATUS AS VARCHAR) AS STATUS
    FROM bureau_history
),
acct AS (
    SELECT
        SK_ID_BUREAU,
        COUNT(*)                                                         AS months,
        SUM(CASE WHEN STATUS IN ('1','2','3','4','5') THEN 1 ELSE 0 END) AS dlq_months,
        SUM(CASE WHEN STATUS IN ('1','2','3','4','5') AND MONTHS_BALANCE >= -12
                 THEN 1 ELSE 0 END)                                      AS dlq_months_12m,
        MAX(CASE WHEN STATUS IN ('1','2','3','4','5')
                 THEN CAST(STATUS AS INTEGER) ELSE 0 END)                AS worst_status
    FROM h
    GROUP BY SK_ID_BUREAU
)
SELECT
    b.SK_ID_CURR,
    COUNT(*)                                             AS BB_ACCOUNTS,
    SUM(a.months)                                        AS BB_MONTHS,
    SUM(a.dlq_months)                                    AS BB_DLQ_MONTHS,
    SUM(a.dlq_months) * 1.0 / NULLIF(SUM(a.months), 0)   AS BB_DLQ_RATE,
    SUM(a.dlq_months_12m)                                AS BB_DLQ_MONTHS_12M,
    SUM(CASE WHEN a.dlq_months > 0 THEN 1 ELSE 0 END)    AS BB_DLQ_ACCOUNTS,
    MAX(a.worst_status)                                  AS BB_WORST_STATUS
FROM acct a
JOIN bureau b USING (SK_ID_BUREAU)
GROUP BY b.SK_ID_CURR
'''

SQL_INSTALMENTS = '''
WITH p AS (
    SELECT
        SK_ID_CURR, DAYS_INSTALMENT, AMT_INSTALMENT, AMT_PAYMENT,
        DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT AS delay
    FROM instalments
    WHERE DAYS_ENTRY_PAYMENT IS NOT NULL
)
SELECT
    SK_ID_CURR,
    COUNT(*)                                                AS INS_COUNT,
    SUM(CASE WHEN delay > 0 THEN 1 ELSE 0 END)              AS INS_LATE_COUNT,
    AVG(CASE WHEN delay > 0 THEN 1.0 ELSE 0 END)            AS INS_LATE_RATE,
    AVG(CASE WHEN delay > 30 THEN 1.0 ELSE 0 END)           AS INS_LATE_30D_RATE,
    AVG(GREATEST(delay, 0))                                 AS INS_DAYS_LATE_MEAN,
    MAX(delay)                                              AS INS_DAYS_LATE_MAX,
    SUM(AMT_PAYMENT) / NULLIF(SUM(AMT_INSTALMENT), 0)       AS INS_PAID_RATIO,
    AVG(CASE WHEN AMT_PAYMENT < AMT_INSTALMENT - 1 THEN 1.0 ELSE 0 END) AS INS_UNDERPAID_RATE,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365
             THEN CASE WHEN delay > 0 THEN 1.0 ELSE 0 END END) AS INS_LATE_RATE_12M
FROM p
GROUP BY SK_ID_CURR
'''

SQL_POS = '''
SELECT
    SK_ID_CURR,
    COUNT(*)                                                     AS POS_MONTHS,
    COUNT(DISTINCT SK_ID_PREV)                                   AS POS_LOANS,
    COUNT(DISTINCT CASE WHEN NAME_CONTRACT_STATUS = 'Completed'
                        THEN SK_ID_PREV END)                     AS POS_COMPLETED_LOANS,
    AVG(CASE WHEN SK_DPD > 0 THEN 1.0 ELSE 0 END)                AS POS_DPD_RATE,
    AVG(CASE WHEN SK_DPD_DEF > 0 THEN 1.0 ELSE 0 END)            AS POS_DPD_DEF_RATE,
    MAX(SK_DPD)                                                  AS POS_DPD_MAX,
    MAX(SK_DPD_DEF)                                              AS POS_DPD_DEF_MAX,
    AVG(CASE WHEN MONTHS_BALANCE >= -12
             THEN CASE WHEN SK_DPD > 0 THEN 1.0 ELSE 0 END END)  AS POS_DPD_RATE_12M
FROM pos
GROUP BY SK_ID_CURR
'''

SQL_CARD = '''
SELECT
    SK_ID_CURR,
    COUNT(*)                                                         AS CC_MONTHS,
    AVG(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0))            AS CC_UTILISATION_MEAN,
    MAX(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0))            AS CC_UTILISATION_MAX,
    AVG(AMT_BALANCE)                                                 AS CC_BALANCE_MEAN,
    MAX(AMT_CREDIT_LIMIT_ACTUAL)                                     AS CC_LIMIT_MAX,
    AVG(AMT_DRAWINGS_ATM_CURRENT)                                    AS CC_ATM_DRAWINGS_MEAN,
    AVG(CASE WHEN SK_DPD > 0 THEN 1.0 ELSE 0 END)                    AS CC_DPD_RATE,
    MAX(SK_DPD)                                                      AS CC_DPD_MAX,
    AVG(CASE WHEN AMT_INST_MIN_REGULARITY > 0
              AND AMT_PAYMENT_CURRENT < AMT_INST_MIN_REGULARITY
             THEN 1.0 ELSE 0 END)                                    AS CC_BELOW_MIN_PAYMENT_RATE
FROM card
GROUP BY SK_ID_CURR
'''

AGGREGATIONS = {
    "PREV": SQL_PREVIOUS,
    "BUR": SQL_BUREAU,
    "BB": SQL_BUREAU_HISTORY,
    "INS": SQL_INSTALMENTS,
    "POS": SQL_POS,
    "CC": SQL_CARD,
}

agg_tables = {}
for prefix, sql in AGGREGATIONS.items():
    table = con.execute(sql).df()
    assert table["SK_ID_CURR"].is_unique, f"{prefix}: aggregation is not one row per applicant"
    agg_tables[prefix] = table
    print(f"{prefix:<5} {table.shape[0]:>7,} applicants x {table.shape[1] - 1:>2} features")

## 4. Clean the application table and engineer applicant-level ratios

Ratios encode affordability in a way that is comparable across income levels, which is how credit analysts reason about applications.

In [ ]:
def clean_applications(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Placeholder value: 365243 days employed means "not employed / pensioner", not 1,000 years.
    if "DAYS_EMPLOYED" in df:
        df["FLAG_DAYS_EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] == DATE_SENTINEL).astype(int)
        df.loc[df["DAYS_EMPLOYED"] == DATE_SENTINEL, "DAYS_EMPLOYED"] = np.nan

    if "CODE_GENDER" in df:
        df.loc[df["CODE_GENDER"] == "XNA", "CODE_GENDER"] = np.nan

    income = df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    annuity = df["AMT_ANNUITY"].replace(0, np.nan)

    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365.25
    df["YEARS_EMPLOYED"] = -df["DAYS_EMPLOYED"] / 365.25
    df["EMPLOYED_TO_AGE_RATIO"] = df["YEARS_EMPLOYED"] / df["AGE_YEARS"]
    df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / income
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / income
    df["CREDIT_TERM_MONTHS"] = df["AMT_CREDIT"] / annuity
    if "AMT_GOODS_PRICE" in df:
        df["GOODS_TO_CREDIT_RATIO"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    if "CNT_FAM_MEMBERS" in df:
        df["INCOME_PER_FAMILY_MEMBER"] = income / df["CNT_FAM_MEMBERS"].replace(0, np.nan)

    ext = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in df]
    df["EXT_SOURCE_MEAN"] = df[ext].mean(axis=1)
    df["EXT_SOURCE_MIN"] = df[ext].min(axis=1)
    df["EXT_SOURCE_MAX"] = df[ext].max(axis=1)
    df["EXT_SOURCE_STD"] = df[ext].std(axis=1)
    df["EXT_SOURCE_N_MISSING"] = df[ext].isna().sum(axis=1)
    for c in ext:
        df[f"{c}_MISSING"] = df[c].isna().astype(int)

    doc_cols = [c for c in df.columns if c.startswith("FLAG_DOCUMENT_")]
    if doc_cols:
        df["DOCUMENTS_PROVIDED"] = df[doc_cols].sum(axis=1)

    return df


train_raw = pd.read_csv(DATA_DIR / FILES["applications"])
assert train_raw["SK_ID_CURR"].is_unique and train_raw["TARGET"].notna().all()

frames = {"train": clean_applications(train_raw)}
if HAS_TEST:
    test_raw = pd.read_csv(DATA_DIR / TEST_FILE)
    assert test_raw["SK_ID_CURR"].is_unique
    overlap = set(test_raw["SK_ID_CURR"]) & set(train_raw["SK_ID_CURR"])
    assert not overlap, f"{len(overlap)} applicants appear in both train and test"
    frames["test"] = clean_applications(test_raw)

print("Days-employed placeholder rows:", int(frames["train"]["FLAG_DAYS_EMPLOYED_ANOMALY"].sum()))
frames["train"][["AGE_YEARS", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM_MONTHS", "EXT_SOURCE_MEAN"]].describe().round(3)

## 5. Merge everything into one modelling table

Left joins keep every applicant. Count features are set to $0$ when an applicant has no records (a true count), while rates stay missing and a `HAS_*` flag records that the history does not exist.

In [ ]:
HISTORY_FLAGS = {
    "PREV": "HAS_PREVIOUS_APPLICATIONS",
    "BUR": "HAS_BUREAU_RECORDS",
    "BB": "HAS_BUREAU_MONTHLY_HISTORY",
    "INS": "HAS_INSTALMENT_HISTORY",
    "POS": "HAS_POS_HISTORY",
    "CC": "HAS_CREDIT_CARD_HISTORY",
}
COUNT_SUFFIXES = ("_COUNT", "_ACCOUNTS", "_MONTHS", "_LOANS", "_12M", "_TYPES")


def build_features(apps: pd.DataFrame) -> pd.DataFrame:
    out = apps
    for prefix, table in agg_tables.items():
        out = out.merge(table, on="SK_ID_CURR", how="left", validate="one_to_one")
        flag = HISTORY_FLAGS[prefix]
        out[flag] = out["SK_ID_CURR"].isin(table["SK_ID_CURR"]).astype(int)
        count_cols = [c for c in table.columns
                      if c != "SK_ID_CURR" and c.endswith(COUNT_SUFFIXES) and "RATE" not in c]
        out[count_cols] = out[count_cols].fillna(0)
    assert len(out) == len(apps) and out["SK_ID_CURR"].is_unique
    return out


features = {name: build_features(df) for name, df in frames.items()}

train_features = features["train"]
print("Modelling table:", train_features.shape)
print("\nHistory coverage and observed repayment difficulty rate:")
coverage = pd.DataFrame({
    flag: train_features.groupby(flag)["TARGET"].agg(["count", "mean"]).stack()
    for flag in HISTORY_FLAGS.values()
}).T
coverage.columns = [f"{'has' if h else 'no'}_{stat}" for h, stat in coverage.columns]
coverage.round(4)

## 6. Save

Parquet keeps the column types (including categoricals and missing values) and is much smaller and faster to load than CSV.

In [ ]:
for name, df in features.items():
    path = OUT_DIR / f"features_{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved {name:<5} {df.shape} -> {path}")

new_cols = sorted(set(train_features.columns) - set(train_raw.columns))
print(f"\n{len(new_cols)} engineered features added to the {train_raw.shape[1]} original columns.")

## Summary for the report

* Seven tables were reduced to **one row per applicant** with SQL aggregations in DuckDB, avoiding the one-to-many duplication that would otherwise inflate the dataset and leak information across rows.
* Known data quality problems (date placeholders, `XNA` codes, negative debt) were fixed before modelling, and each fix is flagged so the model can still learn from the anomaly itself.
* Missing history is represented explicitly. It is informative (thin-file applicants are riskier) and is also a **financial inclusion** issue that the model evaluation in notebook 04 examines.
* No target information was used, so nothing here can leak into model evaluation.